# AI Chatbot for Customer Support 💬🎙️

- *Author : Shravani More*

## 0.2 Feature Extraction & Model Training Workflow File

# 1. Problem Statement
The Chatbot still cannot understand plain text it can only clean the text.

> ### Goal

Teaching the Model to Classify the Intents.

1. Converting **`Text` -> `Numbers`** by (TF-IDF):

 We use **TF-IDF** to transform each cleaned sentence into a row of mathematical weights (a feature vector).

2. **Train a Classifier**:

Feed those numbers + correct labels to an algorithm so it learns the pattern. Then test how well it learned.




# 2. Importing Library

In [157]:
import nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [158]:
import json
import numpy as np
import pandas as pd
import joblib
import textwrap

import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, confusion_matrix


# 3. Reloading the NLP Processor
Same NLP Processor from setup file

In [159]:
# NLP Class
class NLPProcessor:
  def __init__(self):
    self.lemmatizer = WordNetLemmatizer()
    self.stop_words = set(stopwords.words("english"))
    self.stop_words.discard("no")
    self.stop_words.discard("not")

  def clean_text(self, x):
    x =  x.lower()

    # Punctuation
    for i in x:
      if i in string.punctuation:
        x = x.replace(i, "")
    x = "".join([char for char in x if not char.isdigit()])
    x = " ".join(x.split()) # space removing

    return x

  def preprocess_text(self, x):
    x = str(x)

    # Tokenization
    words = word_tokenize(x) # Breaking sentence into individual words.

    # Stopswords + Lemmatization
    cleaned_words = []

    for word in words:
      if word not in self.stop_words: # Removing useless/common words
        lem = self.lemmatizer.lemmatize(word) # Converting to root form
        cleaned_words.append(lem)
    return " ".join(cleaned_words)


  # main function
  def main(self, text):
    x = self.clean_text(text)
    token = self.preprocess_text(x)

    return token

processor = NLPProcessor()

# 4. Loading the intents

Loading intents.json and Extracting Patterns + Labels

Patterns[0] is always paired with labels[0]

In [160]:
with open("/content/intents.json", "r") as f:
  data = json.load(f)
data["intents"][:1]

[{'tag': 'greetings',
  'patterns': ['Hello',
   'Hi',
   'Goodmorning',
   'Goodevening',
   'Hi there',
   'Hey there',
   'Howdy',
   "What 's up",
   'Greetings',
   'Good afternoon',
   'Is anyone there',
   'Hello there',
   'Hi, I need help',
   'Hey Can You help me'],
  'responses': ['Hello How can I Help you today?',
   'Hi there! What can I do for you today?',
   'Hey! I am here to help. What do you need?',
   'Good to see you! How can I assist?']}]

In [161]:
# Extracting all the patterns and labels

patterns = [] # x - input text (features)
labels = [] # y - the correct intent tag (target)

for intent in data["intents"]:
  for p in intent["patterns"]:
    preprocessed = processor.main(p)  # Calling the main method of NLP class on patterns to be cleaned
    patterns.append(preprocessed)
    labels.append(intent["tag"])

# Every single pattern needs its own matching label.

print(f"Total patterns extracted: {len(patterns)}")
print(f"Total Labels extracted: {len(labels)}")
print(f"Unique intent classes: {len(set(labels))}")
# X (features) and y (labels) have same rows.


print(f"\nClass Distribution: ")

for i in data["intents"]:
  print(f"{i["tag"]} : {len(i["patterns"])} patterns")


Total patterns extracted: 140
Total Labels extracted: 140
Unique intent classes: 9

Class Distribution: 
greetings : 14 patterns
order_status : 15 patterns
return_policy : 17 patterns
billing_inquiry : 16 patterns
store_hours : 15 patterns
product_pricing : 15 patterns
escalate_to_human : 15 patterns
chitchat : 16 patterns
shipping_delay : 17 patterns


# 5. TF-IDF Vectorisation
Term Frequency - Inverse Document Frequency : Convert raw text into numerical feature vectors

Term Frequency (TF): Measures how frequently a term appears in a specific document.

Inverse Document Frequency (IDF): Measures how rare or common a term is across the entire corpus. Words like "the" or "is" have low IDF, whereas specialized words have high IDF.

- Builds a vocabulary of the top 500 most important words across all patterns
- Converts each pattern into a vector of 500 numbers
- Each number = TF-IDF score of that word in that pattern
- Higher score = more important/unique word for that pattern


In [162]:
# TFIDF

tfidf = TfidfVectorizer(max_features= 500, ngram_range=(1,2))
x = tfidf.fit_transform(patterns)

print(f"Shape of MAtrix: {x.shape}")
print(f"Total Rows : {x.shape[0]} & Total Columns : {x.shape[1]}")

vb = tfidf.get_feature_names_out()
print("\nSample Vocabulary: ")
print(vb[:5])

# Converting to the dataframe
x_df = pd.DataFrame(x.toarray(), columns= vb)
x_df


Shape of MAtrix: (140, 303)
Total Rows : 140 & Total Columns : 303

Sample Vocabulary: 
['account' 'afternoon' 'agent' 'agent please' 'alright']


,account,afternoon,agent,agent please,alright,amount,amount charged,anyone,arrive,arrived,...,working,working hour,wrong,wrong amount,wrong product,yeah,yeah okay,yep,yesterday,yet
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
136,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
137,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
138,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##  5.1 Saving Vectors file

In [163]:
# Saving the pkl file
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
print("Tfidf_vectorizer.pkl saved")

# Reloading the file to verify
test_load = joblib.load("/content/tfidf_vectorizer.pkl")
print(f"Reloaded File: \n", len(test_load.vocabulary_))

Tfidf_vectorizer.pkl saved
Reloaded File: 
 303


# 6. Train Test Split
- The dataset was split into 80% training and 20% testing data.

- Stratification was applied to maintain the same distribution of intent class in both sets.

In [164]:
# Train Test Split : x = vectors , y = labels

x_train, x_test, y_train, y_test = train_test_split(x, labels, test_size = 0.2, random_state = 42, stratify=labels )


In [165]:
print("xtrain:" , x_train.shape)
print("xtest: ", x_test.shape)
print("ytrain: ", len(y_train))
print("ytest: ", len(y_test))

xtrain: (112, 303)
xtest:  (28, 303)
ytrain:  112
ytest:  28


# 7. Model Training

Baseline Machine Learning models including Naive Bayes, Logistic Regression were implemented to establish initial performance benchmarks.



## 7.1 Model Dictionary

A models dictionary was created to organize and compare multiple algorithms efficiently. It includes `Naive Bayes`, which is simple and fast for text data, `Logistic Regression`, which performs well for linear classification tasks

In [166]:
models = {
    "Naive Bayes" : MultinomialNB(alpha = 0.1),
    "Logistic Regression" : LogisticRegression(max_iter = 1000, random_state = 42, C=1.0)
    }

## 7.2 Model Fitting

Model performance was evaluated using the Accuracy & weighted F1-score, which considers both precision and recall while accounting for class imbalance, providing a balanced measure of overall model effectiveness.

In [167]:
def model_eval(model, xtrain, xtest, ytrain, ytest):
  model.fit(xtrain, ytrain)

  # Predict
  y_pred = model.predict(xtest)

  # Metrics
  acc = accuracy_score(ytest, y_pred)
  report = classification_report(ytest, y_pred, zero_division=0)
  f1 = f1_score(ytest, y_pred, average = "weighted")
  cm = confusion_matrix(ytest, y_pred)

  return acc, report, f1, cm


## 7.4 Model Evaluation

In [168]:
results = []

for name, model in models.items():
  print("="*45)
  print(f"{name}")
  print("="*45)

  acc, report, f1, cm = model_eval(model, x_train, x_test, y_train, y_test)

  # Storing accuracy of each model using name as key
  results.append({
      "Model" : name,
      "Accuracy": acc,
      "F1_Score" : f1
  })


  print(f"Accuracy: {acc:.2f}")
  print("Classification Report: \n", report)
  print("F1 SCore: ", f1)
  print("Confusion Matrix: ")
  print(cm)
  print("\n")


Naive Bayes
Accuracy: 0.68
Classification Report: 
                    precision    recall  f1-score   support

  billing_inquiry       1.00      1.00      1.00         3
         chitchat       0.60      1.00      0.75         3
escalate_to_human       0.60      1.00      0.75         3
        greetings       1.00      0.33      0.50         3
     order_status       0.50      1.00      0.67         3
  product_pricing       0.67      0.67      0.67         3
    return_policy       0.75      1.00      0.86         3
   shipping_delay       1.00      0.25      0.40         4
      store_hours       0.00      0.00      0.00         3

         accuracy                           0.68        28
        macro avg       0.68      0.69      0.62        28
     weighted avg       0.69      0.68      0.61        28

F1 SCore:  0.613265306122449
Confusion Matrix: 
[[3 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0]
 [0 0 3 0 0 0 0 0 0]
 [0 2 0 1 0 0 0 0 0]
 [0 0 0 0 3 0 0 0 0]
 [0 0 0 0 0 2 1 0 0]
 [0 

In [169]:
results

[{'Model': 'Naive Bayes',
  'Accuracy': 0.6785714285714286,
  'F1_Score': 0.613265306122449},
 {'Model': 'Logistic Regression',
  'Accuracy': 0.75,
  'F1_Score': 0.7043367346938776}]

## 7.5 Models Comparsion DF



In [170]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop = True)
results_df

,Model,Accuracy,F1_Score
0,Logistic Regression,0.750000,0.704337
1,Naive Bayes,0.678571,0.613265


## 7.6 Best Performing Model

In [171]:
print("The Best Performing Model is : ", results_df["Model"][0])
print("The Accuracy is : ", results_df["Accuracy"][0])
print("The F1 Score of Model is : ", results_df["F1_Score"][0])

The Best Performing Model is :  Logistic Regression
The Accuracy is :  0.75
The F1 Score of Model is :  0.7043367346938776


# 8. Saving the Best Model

In [172]:
best_model = results_df["Model"][0]
model_to_save = models[best_model]

# Saving Best Model
joblib.dump(model_to_save, "/content/intent_classifier.pkl")
print("intent_classifier.pkl saved ", best_model )


intent_classifier.pkl saved  Logistic Regression


# 10.  Testing the Model

The trained model is tested on a random query to evaluate its prediction capability on unseen data.

This helps verify how well the model generalizes to real-world inputs.

In [173]:
# Random Query Testing
s = "where is my package"
s_preprocess = processor.main(s)
s_vector = tfidf.transform([s_preprocess])
prediction = model_to_save.predict(s_vector)
confidence = model_to_save.predict_proba(s_vector).max()

print("Sample Testing")
print("Input Query: ", s)
print("Prediction: ", prediction[0])
print(f"Confidence: {confidence:.2%}")


Sample Testing
Input Query:  where is my package
Prediction:  shipping_delay
Confidence: 19.54%


In [174]:
test_phrases = [
    "ok fine",
    "alright",
    "got it",
    "sure",
    "ok thanks",
    "where is my package",
    "I want refund",
    "talk to human",
    "My parcel delivery late"
]

print(f"{"Input":<25} {"Predicted Intent":<25} {"Confidence"}")
print("-" * 80)

for phrase in test_phrases:
  p_preprocess = processor.main(phrase)
  p_vector = tfidf.transform([p_preprocess])
  prediction = model_to_save.predict(p_vector)[0]
  confidence = float(model_to_save.predict_proba(p_vector).max())
  status = "High" if confidence > 0.40 else "Low"
  print(f"{phrase:<25} {prediction:<25} {confidence:.2%} {status}")

Input                     Predicted Intent          Confidence
--------------------------------------------------------------------------------
ok fine                   chitchat                  41.78% High
alright                   chitchat                  24.58% Low
got it                    chitchat                  17.56% Low
sure                      chitchat                  24.58% Low
ok thanks                 chitchat                  39.20% Low
where is my package       shipping_delay            19.54% Low
I want refund             return_policy             22.85% Low
talk to human             escalate_to_human         29.56% Low
My parcel delivery late   shipping_delay            17.05% Low
